# 11 — Bayesian Inference
**References:** Gelman et al. (2013) Ch. 1–6 · Casella & Berger (2002) Ch. 7 · Robert (2007)

## Narrative thread
```
Prior -> Likelihood -> Posterior -> Conjugate families -> Jeffreys prior -> MCMC -> Bayesian vs Frequentist
```

## 1. Bayes' theorem for inference

$$\underbrace{\pi(\theta \mid \mathbf{x})}_{\text{posterior}} = \frac{\overbrace{f(\mathbf{x} \mid \theta)}^{\text{likelihood}} \cdot \overbrace{\pi(\theta)}^{\text{prior}}}{\underbrace{\int f(\mathbf{x} \mid \theta) \pi(\theta)\,d\theta}_{\text{marginal likelihood}}} \propto f(\mathbf{x} \mid \theta)\,\pi(\theta)$$

The posterior encodes **all information** about $\theta$ after seeing data $\mathbf{x}$.

**Point estimates from the posterior:**
- **MAP** (maximum a posteriori): $\hat\theta_{\text{MAP}} = \arg\max_\theta \pi(\theta\mid\mathbf{x})$
- **Posterior mean:** $E[\theta\mid\mathbf{x}]$ — minimizes posterior expected squared error
- **Posterior median:** minimizes posterior expected absolute error

**Credible intervals (HDI/HPD):** $[a,b]$ such that $P(\theta \in [a,b]\mid\mathbf{x}) = 1-\alpha$

> **Bayesian vs Frequentist:** The posterior IS a probability for $\theta$ conditional on data. No repeated sampling needed — the prior represents beliefs before seeing data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Beta-Binomial conjugate model: updating beliefs about p ───────────────
# Prior: p ~ Beta(alpha0, beta0)
# Likelihood: X | p ~ Binomial(n, p)
# Posterior: p | X=x ~ Beta(alpha0 + x, beta0 + n - x)

alpha0, beta0 = 2, 5        # prior: believe p is small
n_coin        = 20
p_true        = 0.6
x_obs         = np.random.binomial(n_coin, p_true)  # observed successes

alpha_post = alpha0 + x_obs
beta_post  = beta0  + n_coin - x_obs

ps = np.linspace(0, 1, 400)
prior_pdf     = stats.beta.pdf(ps, alpha0, beta0)
posterior_pdf = stats.beta.pdf(ps, alpha_post, beta_post)
likelihood_unnorm = stats.binom.pmf(x_obs, n_coin, ps)
likelihood_unnorm = likelihood_unnorm / likelihood_unnorm.max() * posterior_pdf.max()

post_mean   = alpha_post / (alpha_post + beta_post)
post_mode   = (alpha_post - 1) / (alpha_post + beta_post - 2)
ci_lo, ci_hi = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(ps, prior_pdf,      color='#06d6a0', lw=2.5, label=f'Prior: Beta({alpha0},{beta0})')
ax.plot(ps, likelihood_unnorm, color='gray',  lw=2, linestyle='--', label=f'Likelihood (scaled), x={x_obs}')
ax.plot(ps, posterior_pdf,  color='#4361ee', lw=2.5, label=f'Posterior: Beta({alpha_post},{beta_post})')
ax.axvline(p_true, color='#f72585', lw=2, linestyle=':', label=f'True p={p_true}')
ax.axvline(post_mean, color='#4361ee', lw=1.5, linestyle='--', alpha=0.7, label=f'Post. mean={post_mean:.3f}')
ax.fill_between(ps, posterior_pdf, where=(ps>=ci_lo)&(ps<=ci_hi), alpha=0.2, color='#4361ee',
                label=f'95% HDI [{ci_lo:.3f},{ci_hi:.3f}]')
ax.set_xlabel('p'); ax.set_ylabel('Density')
ax.set_title(f'Beta-Binomial: n={n_coin}, x={x_obs} successes\n'
              'Posterior = updated belief after seeing data')
ax.legend(fontsize=7)

# Sequential updating
ax2 = axes[1]
ns_seq = [0, 5, 20, 50, 200]
cum_x  = 0
cum_n  = 0
colors_seq = plt.cm.Blues(np.linspace(0.3, 1.0, len(ns_seq)))

for i, n_add in enumerate(ns_seq):
    if n_add > 0:
        new_x  = np.random.binomial(n_add, p_true)
        cum_x += new_x; cum_n += n_add
    a_post = alpha0 + cum_x
    b_post = beta0  + cum_n - cum_x
    ax2.plot(ps, stats.beta.pdf(ps, a_post, b_post),
             lw=2, color=colors_seq[i],
             label=f'n={cum_n}: Beta({a_post},{b_post})')

ax2.axvline(p_true, color='#f72585', lw=2, linestyle=':', label=f'True p={p_true}')
ax2.set_xlabel('p'); ax2.set_ylabel('Density')
ax2.set_title('Sequential Bayesian updating\nPosterior concentrates around true p as n grows')
ax2.legend(fontsize=7)

plt.suptitle('Bayesian Inference: Beta-Binomial Conjugate Model', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Conjugate families

A prior $\pi(\theta)$ is **conjugate** to the likelihood $f(\mathbf{x}\mid\theta)$ if the posterior $\pi(\theta\mid\mathbf{x})$ is in the same family as the prior.

| Likelihood | Conjugate prior | Posterior update |
|---|---|---|
| Binomial($n,p$) | Beta($\alpha_0, \beta_0$) | Beta($\alpha_0+x$, $\beta_0+n-x$) |
| Poisson($\lambda$) | Gamma($\alpha_0, \beta_0$) | Gamma($\alpha_0+\sum x_i$, $\beta_0+n$) |
| Normal($\mu, \sigma^2$) known $\sigma^2$ | Normal($\mu_0, \tau^2$) | Normal($\mu_n, \tau_n^2$) |
| Normal($\mu, \sigma^2$) unknown both | Normal-Inverse-Gamma | Normal-Inverse-Gamma |
| Multinomial($\mathbf{p}$) | Dirichlet($\boldsymbol{\alpha}$) | Dirichlet($\boldsymbol{\alpha} + \mathbf{x}$) |

For Normal with known $\sigma^2$: $\mu \sim \mathcal{N}(\mu_0, \tau^2)$, then:
$$\mu_n = \frac{\tau^{-2}\mu_0 + n\sigma^{-2}\bar x}{\tau^{-2} + n\sigma^{-2}}, \qquad \tau_n^{-2} = \tau^{-2} + n\sigma^{-2}$$

The posterior mean is a **precision-weighted average** of prior mean and data mean.

## 3. Non-informative and Jeffreys priors

**Jeffreys prior:** $\pi(\theta) \propto \sqrt{I(\theta)}$ — invariant under reparameterization.

| Model | Jeffreys prior |
|---|---|
| Binomial($p$) | Beta$(1/2, 1/2)$ |
| Poisson($\lambda$) | $\pi(\lambda) \propto \lambda^{-1/2}$ = Gamma$(1/2, 0)$ |
| Normal mean $\mu$ ($\sigma$ known) | $\pi(\mu) \propto 1$ (flat/improper) |
| Normal variance $\sigma^2$ | $\pi(\sigma^2) \propto 1/\sigma^2$ |

**Bernstein-von Mises theorem:** Under regularity conditions, the posterior concentrates around the MLE:
$$\pi(\theta \mid \mathbf{x}) \approx \mathcal{N}\!\left(\hat\theta_{\text{MLE}},\, \frac{1}{n I(\hat\theta)}\right) \quad \text{as } n \to \infty$$

Bayesian and frequentist inference agree in large samples.

In [ ]:
# ── MCMC: Metropolis-Hastings for non-conjugate posterior ─────────────────
# Model: X ~ Cauchy(theta, 1),  prior: theta ~ N(0, 10^2)
# Posterior: prop to prod_i f(xi|theta) * pi(theta)

x_data = np.array([1.2, 2.5, 1.8, 3.1, 0.9, 2.2, 1.5])  # Cauchy data

def log_posterior(theta, x, prior_sd=10):
    log_lik   = np.sum(stats.cauchy.logpdf(x, theta, 1))
    log_prior = stats.norm.logpdf(theta, 0, prior_sd)
    return log_lik + log_prior

# Metropolis-Hastings
n_mcmc   = 50_000
burn_in  = 5_000
proposal_sd = 0.5

chain    = [1.0]
accepted = 0

for _ in range(n_mcmc - 1):
    current   = chain[-1]
    proposed  = current + np.random.normal(0, proposal_sd)
    log_ratio = log_posterior(proposed, x_data) - log_posterior(current, x_data)
    if np.log(np.random.uniform()) < log_ratio:
        chain.append(proposed)
        accepted += 1
    else:
        chain.append(current)

chain = np.array(chain[burn_in:])
accept_rate = accepted / n_mcmc

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(chain[:500], color='#4361ee', lw=0.8, alpha=0.7)
axes[0].set_title(f'MCMC trace (first 500 post-burnin)\nAccept rate={accept_rate:.2f}')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('θ')

axes[1].hist(chain, bins=80, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
# Overlay MLE estimate
from scipy.optimize import minimize_scalar
neg_ll = lambda t: -np.sum(stats.cauchy.logpdf(x_data, t, 1))
mle_theta = minimize_scalar(neg_ll, bounds=(-5, 5), method='bounded').x
axes[1].axvline(mle_theta, color='#f72585', lw=2, linestyle='--', label=f'MLE={mle_theta:.2f}')
axes[1].axvline(chain.mean(), color='#06d6a0', lw=2, label=f'Post. mean={chain.mean():.2f}')
axes[1].set_title('Posterior distribution (MCMC)\nCauchy likelihood, Normal prior')
axes[1].set_xlabel('θ'); axes[1].legend(fontsize=8)

# Autocorrelation
lags = np.arange(0, 100)
acf  = [np.corrcoef(chain[:-k if k>0 else len(chain)], chain[k:])[0,1] for k in lags]
axes[2].bar(lags, acf, color='#4361ee', alpha=0.7, width=0.8)
axes[2].axhline(0, color='k', lw=1)
axes[2].axhline(1.96/np.sqrt(len(chain)), color='#f72585', lw=1.5, linestyle='--', label='±95% band')
axes[2].axhline(-1.96/np.sqrt(len(chain)), color='#f72585', lw=1.5, linestyle='--')
axes[2].set_title('MCMC autocorrelation\nLow autocorrelation = good mixing')
axes[2].set_xlabel('Lag'); axes[2].legend(fontsize=8)

plt.suptitle('Metropolis-Hastings MCMC: Cauchy likelihood, Normal prior', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

ci_95 = np.percentile(chain, [2.5, 97.5])
print(f'\n95% Credible Interval: [{ci_95[0]:.3f}, {ci_95[1]:.3f}]')
print(f'Posterior mean: {chain.mean():.3f}')
print(f'MLE:            {mle_theta:.3f}')

## 4. Bayesian vs Frequentist comparison

| Aspect | Frequentist | Bayesian |
|---|---|---|
| $\theta$ | Fixed unknown | Random variable |
| Data | Random | Fixed (observed) |
| Confidence interval | $P$(interval contains $\theta$) = $1-\alpha$ | $P$(θ in interval $\mid$ data) = $1-\alpha$ |
| Prior | None | Required — encodes beliefs |
| Large-sample behavior | MLE is efficient | Posterior ≈ MLE + Cramér-Rao (BvM theorem) |
| Prediction | Plug-in MLE | Integrate over posterior |

**Key takeaway:** Bayesian inference gives full probability distributions for parameters, naturally handles uncertainty propagation, and supports direct probability statements about $\theta$. The cost is the need to specify a prior.

**Next:** notebook 12 — nonparametric inference: bootstrap theory and rank-based tests.